### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
from scipy.special import expit

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

In [3]:
def one_sided_posterior_prob(idata, predictor, direction='greater'):
    """Calculate the one-sided posterior probability that the samples are greater than zero."""
    
    beta = idata.posterior[predictor].values.flatten()
    if direction == 'greater':
        prob = np.mean(beta > 0)
    elif direction == 'less':
        prob = np.mean(beta < 0)
    else:
        raise ValueError("Direction must be 'greater' or 'less'.")
    return prob

def odds_ratio_summary(idata, predictor):
    """Calculate the odds ratio summary statistics for a given predictor."""
    beta = idata.posterior[predictor].values.flatten()
    or_mean = np.exp(beta).mean()
    or_low = np.percentile(np.exp(beta), 2.5)
    or_high = np.percentile(np.exp(beta), 97.5)
    return or_mean, or_low, or_high

def posterior_table(idata, predictors, directions):
    """Summarize the model by calculating posterior probabilities for each predictor."""
    table = PrettyTable()
    table.field_names = ["Predictor", "direction", "P"]
    
    for predictor, direction in zip(predictors, directions):
        prob = one_sided_posterior_prob(idata, predictor, direction)
        table.add_row([predictor, direction, f"{prob:.3f}"])
    
    return table

def OR_table(idata, predictors):
    """Summarize the model by calculating posterior probabilities and odds ratios for each predictor."""
    table = PrettyTable()
    table.field_names = ["Predictor", "Mean", "Lower (2.5%)", "Upper (97.5%)"]
    
    for predictor in predictors:
        or_mean, or_low, or_high = odds_ratio_summary(idata, predictor)
        table.add_row([predictor, f"{or_mean:.3f}", f"{or_low:.3f}", f"{or_high:.3f}"])
    
    return table


def compute_mediation(idata_mediator, idata_full, 
                      delta_CH=0.0, delta_GC=0.0, 
                      baseline_intercept=None,
                      mediator_prefix="", outcome_prefix=""):
    """
    Compute Bayesian mediation effects for a contrast Δx on (CH, GC, CHxGC),
    using posterior draws from:
      - mediator model:  Synchrony ~ 1 + CH * GC  (+ RE; we use population-level)
      - full outcome model: Correct ~ 1 + Synchrony + CH * GC  (+ RE; population-level)

    Parameters
    ----------
    idata_mediator : arviz.InferenceData
        Posterior from mediator model (gaussian).
    idata_full : arviz.InferenceData
        Posterior from full (outcome) model (bernoulli-logit).
    delta_CH : float
        Contrast for CH (e.g., +1 SD).
    delta_GC : float
        Contrast for GC (e.g., +1 SD).
    baseline_intercept : float or None
        If None, uses posterior draws of Intercept from the full model.
        You can pass a fixed value to standardize baseline if desired.
    mediator_prefix : str
        Optional prefix if variable names are namespaced (rare).
    outcome_prefix : str
        Optional prefix if variable names are namespaced (rare).

    Returns
    -------
    summary : pd.DataFrame
        Posterior summaries (mean, hdi) for IE, DE, TE, plus baseline p0.
    draws : pd.DataFrame
        Draw-wise IE, DE, TE (for custom plotting if needed).
    """
    # ---------- Helper: extract posterior draws to tidy pd.Series ----------
    def extract(series_name, idata):
        # az.extract returns a DataFrame with one column per var; pick by name
        df = az.extract(idata, var_names=[series_name]).to_dataframe()
        # If multiple columns (e.g., coords), sum along columns (shouldn't happen for these names)
        if df.shape[1] > 1:
            return df.sum(axis=1)
        return df.iloc[:, 0]

    # ---------- Build variable names (as in your Bambi output) ----------
    # Mediator (gaussian): Synchrony ~ Intercept + CH + GC + CH:GC
    m_Intercept = extract(f"{mediator_prefix}Intercept", idata_mediator)
    a_CH        = extract(f"{mediator_prefix}ContrastHeterogeneity", idata_mediator)
    a_GC        = extract(f"{mediator_prefix}GridCoarseness", idata_mediator)
    a_CHxGC     = extract(f"{mediator_prefix}ContrastHeterogeneity:GridCoarseness", idata_mediator)

    # Outcome (bernoulli-logit): Correct ~ Intercept + Syn + CH + GC + CH:GC
    o_Intercept = extract(f"{outcome_prefix}Intercept", idata_full)
    b_Syn       = extract(f"{outcome_prefix}Synchrony", idata_full)
    c_CH        = extract(f"{outcome_prefix}ContrastHeterogeneity", idata_full)
    c_GC        = extract(f"{outcome_prefix}GridCoarseness", idata_full)
    c_CHxGC     = extract(f"{outcome_prefix}ContrastHeterogeneity:GridCoarseness", idata_full)

    # ---------- Define the contrast Δx ----------
    # Interaction contrast at the mean of the other regressor: Δ(CHxGC) = ΔCH * E[GC] + ΔGC * E[CH] + cross-term.
    # With z-scored predictors and evaluating at means, use ΔCHxGC = 0 for main-effect contrasts.
    delta_CHxGC = 0.0

    # ---------- Mediated change in Synchrony ----------
    delta_Syn = a_CH * delta_CH + a_GC * delta_GC + a_CHxGC * delta_CHxGC

    # ---------- Baseline linear predictor (population-level) ----------
    eta0 = o_Intercept if baseline_intercept is None else float(baseline_intercept)
    p0 = expit(eta0)

    # ---------- Changes on the logit scale ----------
    delta_eta_direct = c_CH * delta_CH + c_GC * delta_GC + c_CHxGC * delta_CHxGC
    delta_eta_total  = delta_eta_direct + b_Syn * delta_Syn

    # ---------- Convert to probabilities ----------
    p_direct = expit(eta0 + delta_eta_direct)
    p_total  = expit(eta0 + delta_eta_total)

    # ---------- Effects on probability scale ----------
    IE = p_total - p_direct      # Indirect (via Syn)
    DE = p_direct - p0           # Direct (not via Syn)
    TE = p_total  - p0           # Total

    draws = pd.DataFrame({
        "p0": p0.values if hasattr(p0, "values") else p0,
        "IE": IE.values,
        "DE": DE.values,
        "TE": TE.values
    })

    # ---------- Summaries ----------
    def hdi(x, prob=0.95):
        return az.hdi(x, hdi_prob=prob).to_numpy()

    def summarize(name, x):
        lo, hi = hdi(x)
        return pd.Series({
            "mean": np.mean(x),
            "hdi_2.5%": lo,
            "hdi_97.5%": hi
        }, name=name)

    summary = pd.concat([
        summarize("p0", draws["p0"]),
        summarize("IE", draws["IE"]),
        summarize("DE", draws["DE"]),
        summarize("TE", draws["TE"]),
        pd.Series({"prop_mediated_mean": np.mean(draws["IE"] / np.where(draws["TE"]==0, np.nan, draws["TE"]))},
                  name="proportion_mediated")
    ], axis=1).T

    return summary, draws

### Load and prepare data

In [4]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [5]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()


data = load_data(emp_at_file)
# Get the data for the first session
data = get_session_data(data, 1)

# Map synchrony values to each condition in DataFrame
data['Synchrony'] = data['Condition'].apply(lambda x: sync_results_vector[x-1])

data = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])

In [6]:
# Features-only hierarchical logistic regression
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

# Model mediation (Synchrony as function of features)
model_mediator = bmb.Model(
    "Synchrony ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness | SubjectID)",
    data=data,
    family="gaussian"
)

# Model synchrony (mechanism)
model_sync = bmb.Model(
    "Correct ~ 1 + Synchrony + (1 + Synchrony | SubjectID)",
    data=data,
    family="bernoulli"
)

# Full model (features + mechanism)
model_full = bmb.Model(
    "Correct ~ 1 + Synchrony + ContrastHeterogeneity * GridCoarseness + (1 + Synchrony + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

Are the factors that determine synchrony among coupled oscillators (frequency detuning and coupling strength) predictive of human ability to segregate a rectangular figure from its background in texture stimuli? 

In [7]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
/home/mario/miniconda3/envs/bat_env/lib/python3.13/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneit

In [8]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['less', 'less', 'greater']

posterior = posterior_table(idata_features, predictors, directions)
odds_ratios = OR_table(idata_features, predictors)
print(posterior)
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.999 |
|            GridCoarseness            |    less   | 0.998 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 0.999 |
+--------------------------------------+-----------+-------+
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.555 |    0.407     |     0.746     |
|            GridCoarseness            | 0.769 |    0.671     |     0.880     |
| ContrastHeterogeneity:GridCoarseness | 1.266 |    1.128     |     1.428     |
+--------------------------------------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.599,0.152,-0.900,-0.294,0.003,0.003,2455.0,3806.0,1.0
GridCoarseness,-0.266,0.068,-0.400,-0.130,0.001,0.002,3737.0,4094.0,1.0
ContrastHeterogeneity:GridCoarseness,0.234,0.059,0.119,0.354,0.001,0.001,4480.0,3690.0,1.0


Does the synchronization behavior of a biophysical model of V1 predict human ability to segregate a rectangular figure from its background in texture stimuli?

In [9]:
idata_sync = model_sync.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 198 seconds.


In [10]:
predictors = ["Synchrony"]
directions = ['greater']

posterior = posterior_table(idata_sync, predictors, directions)
odds_ratios = OR_table(idata_sync, predictors)
print(posterior)
print(odds_ratios)

az.summary(idata_sync, var_names=predictors, hdi_prob=0.95)

+-----------+-----------+-------+
| Predictor | direction |   P   |
+-----------+-----------+-------+
| Synchrony |  greater  | 0.997 |
+-----------+-----------+-------+
+-----------+-------+--------------+---------------+
| Predictor |  Mean | Lower (2.5%) | Upper (97.5%) |
+-----------+-------+--------------+---------------+
| Synchrony | 2.187 |    1.364     |     3.351     |
+-----------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Synchrony,0.757,0.225,0.309,1.209,0.004,0.004,2515.0,2871.0,1.0


Does model synchrony add predictive power beyond the experimentally manipulated stimulus features? 

In [11]:
idata_full = model_full.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 297 seconds.


In [12]:
# Compare models (LOO)
az.compare({
    "stimulus features": idata_features,
    "synchrony": idata_sync,
    "full model": idata_full,
}, method="BB-pseudo-BMA")

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
full model,0,-3581.934316,32.524524,0.000000,6.001101e-01,31.038143,0.000000,False,log
stimulus features,1,-3582.955598,28.349770,1.021282,3.998899e-01,30.999982,3.038171,False,log
synchrony,2,-3632.505319,16.631833,50.571003,4.213488e-10,29.472196,10.892217,False,log


In [13]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness", "Synchrony"]
directions = ['less', 'less', 'greater', 'greater']

posterior = posterior_table(idata_full, predictors, directions)
odds_ratios = OR_table(idata_full, predictors)
print(posterior)
print(odds_ratios)

az.summary(idata_full, var_names=predictors, hdi_prob=0.95)

+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.999 |
|            GridCoarseness            |    less   | 0.998 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 0.999 |
|              Synchrony               |  greater  | 0.876 |
+--------------------------------------+-----------+-------+
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.599 |    0.458     |     0.754     |
|            GridCoarseness            | 0.790 |    0.694     |     0.896     |
| ContrastHeterogeneity:GridCoarseness | 1.228 |    1.104     |     1.365     |
|              Synchrony        

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.521,0.125,-0.784,-0.286,0.002,0.002,4021.0,4418.0,1.0
GridCoarseness,-0.237,0.066,-0.371,-0.116,0.001,0.001,5565.0,4735.0,1.0
ContrastHeterogeneity:GridCoarseness,0.204,0.053,0.099,0.310,0.001,0.001,8443.0,5780.0,1.0
Synchrony,0.147,0.140,-0.139,0.424,0.002,0.002,5294.0,4397.0,1.0


In [45]:
model_mediator = bmb.Model(
    "Synchrony ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness | SubjectID)",
    data=data,
    family="gaussian"
)

In [ ]:
idata_mediator = model_mediator.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

In [44]:

from sklearn.metrics import r2_score  # just for the R^2 formula

# Prepare design matrix (z-scored already)
CH = data["ContrastHeterogeneity"].to_numpy()
GC = data["GridCoarseness"].to_numpy()
CHxGC = CH * GC
X = np.c_[np.ones_like(CH), CH, GC, CHxGC]   # columns: Intercept, CH, GC, CHxGC
y = data["Synchrony"].to_numpy()

# Pull posterior draws of fixed effects from the mediator model:
# Synchrony ~ 1 + CH * GC + (1 + CH * GC | SubjectID), family=gaussian
def get_draw(name):
    return az.extract(idata_mediator, var_names=[name]).to_dataframe().iloc[:, -1].to_numpy()

beta0 = get_draw("Intercept")
b_CH  = get_draw("ContrastHeterogeneity")
b_GC  = get_draw("GridCoarseness")
b_INT = get_draw("ContrastHeterogeneity:GridCoarseness")

# Stack coefficients into (n_draws, 4)
B = np.column_stack([beta0, b_CH, b_GC, b_INT])  # shape (S, 4)

# Posterior-predictive mean for each draw at population level: yhat_s = X @ B_s
# Vectorized: (S, N) = (S,4) @ (4,N)
yhat = B @ X.T                                      # shape (S, N)

# Compute R^2 for each draw
r2_draws = np.array([r2_score(y, yhat_s) for yhat_s in yhat])

# Summarize Bayesian R^2
r2_mean = float(np.mean(r2_draws))
r2_hdi  = az.hdi(r2_draws, hdi_prob=0.95)
print(f"Bayesian R^2 (population-level): mean={r2_mean:.3f}, 95% CrI=[{r2_hdi[0]:.3f}, {r2_hdi[1]:.3f}]")


Bayesian R^2 (population-level): mean=0.798, 95% CrI=[0.797, 0.798]


### Sensitivity Analysis

In [ ]:
with open('../config/simulation/simulation.toml', 'rb') as f:
    sim_config = tomllib.load(f)

seed = sim_config['random_seed']
rng = np.random.default_rng(seed)

In [ ]:
num_repetitions = 100
effect_sizes = np.linspace(0.3, 0.9, 7) # log-odds

formula = "Correct ~ 1 + Synchrony + (1|SubjectID) + (0 + Synchrony|SubjectID)"
effect_of_interest = "Synchrony"

In [ ]:
subject_index, num_subjects = create_subject_index(data)
synchrony = data["Synchrony"].to_numpy()

# Extract posteriors for intercept and beta_sync as well as for the subject-level random effects
posteriors = az.extract(idata_pure, combined=True)

intercept = np.median(posteriors["Intercept"].to_numpy())
beta_sync = np.median(posteriors["Synchrony"].to_numpy())

sdev_intercept = np.median(posteriors["1|SubjectID_sigma"].to_numpy())              # random intercept SD
sdev_beta_sync = np.median(posteriors["Synchrony|SubjectID_sigma"].to_numpy())    # random slope SD


results = []
table = PrettyTable()
table.field_names = ["Effect Size (log-odds)", "Detection Rate"]
for effect_size in effect_sizes:
    detections = 0
    for r in range(num_repetitions):
        simulated_df = simulate_correct(data, effect_size, synchrony, intercept, sdev_beta_sync, sdev_intercept, subject_index, num_subjects, rng)
        detected = fit_and_decide(simulated_df, formula, effect_of_interest, draws=1000,
                   tune=1000, threshold=0.975)
        detections += int(detected)
    detection_rate = detections / num_repetitions
    results.append({"true_beta_sync": effect_size, "detect_rate": detection_rate})
    table.add_row([effect_size, detection_rate])

print(table)
